# Como funciona a simulação
### Guia para explicar a demonstração de *Observabilidade em ML e Atualização Online*

Este notebook **descreve passo a passo** a simulação usada no vídeo: o que é simulado, por que, e como ler o resultado.
Cada seção tem um quadro **🎤 Fale:** com uma sugestão do que dizer.

---
## 0. A pergunta que a simulação responde
> **O que acontece com um modelo em produção quando o mundo muda, e o que muda se a gente observar e atualizar o modelo?**

Para responder, simulamos um **banco** com um modelo que bloqueia transações suspeitas de cartão de crédito, durante **150 horas**.

| | |
|---|---|
| 🔴 **Modelo estático** | Treinado uma vez, **nunca mais atualizado**. É o "e se ninguém cuidar do modelo?" |
| 🟢 **Modelo online** | Mesmo modelo, mas **aprende com cada lote** quando os rótulos (chargebacks) chegam |
| 🔁 **Mesmas transações** | Os dois recebem exatamente os mesmos dados, para a comparação ser justa |

### O que acontece a cada hora simulada
```
 ┌────────────┐   ┌──────────────┐   ┌───────────────────┐   ┌─────────────────────┐   ┌────────────────┐
 │ 1. Chegam  │ → │ 2. Os 2      │ → │ 3. Medimos os     │ → │ 4. Rótulos chegam   │ → │ 5. Alertas e   │
 │ 250 compras│   │ modelos      │   │ sinais SEM rótulo │   │ (4h depois): medimos│   │ ações (online  │
 │ (novas)    │   │ decidem      │   │ (PSI, % bloqueio) │   │ recall; o ONLINE    │   │ aprende mais   │
 │            │   │ bloquear/não │   │                   │   │ aprende (partial_fit)│  │ rápido)        │
 └────────────┘   └──────────────┘   └───────────────────┘   └─────────────────────┘   └────────────────┘
```

> 🎤 **Fale:** "Simulei um banco com um modelo de fraude durante 150 horas. Coloquei dois modelos lado a lado, com as mesmas transações: um estático, que nunca é atualizado, e um online, que aprende. A cada hora acontece este ciclo: chegam compras, os modelos decidem, a gente mede sinais, os rótulos chegam depois e o online aprende."

## 1. Os parâmetros (o "cenário")
| Parâmetro | Valor | O que significa |
|---|---|---|
| `BATCH` | 250 | Compras que chegam **por hora** |
| `N_STEPS` | 150 | Horas simuladas |
| `BF_START`–`BF_END` | 50–90 | **Black Friday** (data drift) |
| `NEW_FRAUD_AT` | 90 | A partir daqui, **golpistas mudam de tática** (concept drift) |
| `LABEL_DELAY` | 4 | O rótulo (chargeback) chega **4 horas depois** |
| `PSI_LIMIT` | 0,20 | Acima disso: "o perfil dos dados mudou" |
| `RECALL_SLO` | 0,70 | **Meta mínima**: capturar pelo menos 70% das fraudes |
| `FP_COST` | R$ 30 | Custo de **bloquear um cliente bom** (atrito, atendimento) |
| `FRAUD_FIXED_COST` | R$ 120 | Custo fixo de **uma fraude que passou** (chargeback + operação), além do valor da compra |

> ⚠️ Todos esses números são **inventados**, só para ilustrar. Em um banco real viriam de dados reais.

> 🎤 **Fale:** "O cenário é controlado por poucos parâmetros. O mais importante: o rótulo demora 4 horas para chegar, como um chargeback, e eu defini uma meta mínima de 70% de fraudes capturadas. Os valores em reais são inventados."

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from sklearn.linear_model import SGDClassifier

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

SEED = 11
FEATURES = ["log_valor", "madrugada", "log_dist_km", "tx_na_hora"]
BATCH = 250                    # transações por hora
N_STEPS = 150                  # horas simuladas
BF_START, BF_END = 50, 90      # Black Friday
NEW_FRAUD_AT = 90              # golpistas mudam de tática
LABEL_DELAY = 4                # o chargeback chega 4 horas depois
WINDOW = 4
PSI_LIMIT = 0.20               # PSI > 0.2 = "o perfil dos dados mudou"
RECALL_SLO = 0.70              # meta mínima: capturar 70% das fraudes
BLOCK_RATE_FACTOR = 1.8
FP_COST = 30.0                 # R$ por cliente bom bloqueado
FRAUD_FIXED_COST = 120.0       # R$ de chargeback + operação por fraude que passou
LR_NORMAL, LR_BOOST, BOOST_STEPS = 0.01, 0.06, 10

C_S, C_O = "#e74c3c", "#16a085"   # vermelho = estático, verde = online
ACTS = [(0, BF_START, "Ato 1: dia normal", "#2ecc71"),
        (BF_START, BF_END, "Ato 2: Black Friday", "#f39c12"),
        (NEW_FRAUD_AT, N_STEPS, "Ato 3: nova fraude", "#e74c3c")]

## 2. A linha do tempo da história (3 atos)
- **Ato 1 (horas 0–50):** dia normal.
- **Ato 2 (horas 50–90):** **Black Friday** → os *clientes bons* mudam de comportamento (compram mais, mais caro, mais rápido). **A fraude é a mesma.** → *data drift*.
- **Ato 3 (horas 90–150):** **golpistas mudam de tática** → "teste de cartão": valores minúsculos, de madrugada, várias tentativas. → *concept drift*.

> 🎤 **Fale:** "A história tem três atos. No segundo, só o comportamento dos clientes bons muda: é data drift. No terceiro, a própria fraude muda: é concept drift. Isso vai mostrar dois tipos diferentes de problema."

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.4))
for s, e, nome, col in ACTS:
    ax.axvspan(s, e, color=col, alpha=0.25)
    ax.text((s + e) / 2, 0.5, nome, ha="center", va="center", fontweight="bold")
ax.axvline(BF_START, color="k", lw=1); ax.axvline(NEW_FRAUD_AT, color="k", lw=1)
ax.text(BF_START, -0.1, "hora 50\ndata drift", ha="center", va="top", fontsize=9)
ax.text(NEW_FRAUD_AT, -0.1, "hora 90\nconcept drift", ha="center", va="top", fontsize=9)
ax.set_xlim(0, N_STEPS); ax.set_ylim(0, 1); ax.set_yticks([]); ax.set_xlabel("tempo (horas)"); ax.grid(False)
plt.show()

## 3. Os dados: o que é uma transação?
Cada compra tem **4 características**:

| Característica | Significado |
|---|---|
| `log_valor` | Valor da compra (em escala logarítmica; o valor em R$ é `exp(log_valor)`) |
| `madrugada` | 1 se foi de madrugada, 0 caso contrário |
| `log_dist_km` | Distância entre a compra e a casa do cliente (log) |
| `tx_na_hora` | Quantas transações esse cartão fez na última hora |

E um **rótulo**: `1` = fraude, `0` = compra legítima (só conhecido **depois**, via chargeback).

Há **4 tipos de transação** na simulação:
1. **Cliente normal**
2. **Cliente na Black Friday** (valores e frequência maiores)
3. **Fraude clássica** (valor alto, longe de casa, várias tentativas) → Atos 1 e 2
4. **Fraude nova, "teste de cartão"** (valor minúsculo, madrugada, muitas tentativas) → Ato 3

In [ ]:
class Bank:
    def __init__(self, rng):
        self.rng = rng

    def legit(self, n, black_friday):
        r = self.rng
        lv = r.normal(4.0 + (0.8 if black_friday else 0.0), 0.8, n)
        mad = r.random(n) < (0.18 if black_friday else 0.05)
        ld = r.normal(1.0, 0.8, n)
        tx = r.poisson(2.6 if black_friday else 1.0, n)
        return np.column_stack([lv, mad, ld, tx])

    def fraud_classic(self, n):
        r = self.rng
        return np.column_stack([r.normal(6.0, 0.8, n), r.random(n) < 0.5,
                                r.normal(3.0, 0.8, n), r.poisson(3.0, n)])

    def fraud_card_testing(self, n):          # a fraude NOVA do Ato 3
        r = self.rng
        return np.column_stack([r.normal(2.0, 0.5, n), r.random(n) < 0.85,
                                r.normal(1.2, 0.8, n), r.poisson(5.0, n)])

    def batch(self, step, n=BATCH):
        bf = BF_START <= step < BF_END
        new = step >= NEW_FRAUD_AT
        n_fraud = self.rng.binomial(n, 0.08 if new else 0.06)
        Xl = self.legit(n - n_fraud, bf)
        Xf = self.fraud_card_testing(n_fraud) if new else self.fraud_classic(n_fraud)
        X = np.vstack([Xl, Xf]).astype(float)
        y = np.concatenate([np.zeros(n - n_fraud), np.ones(n_fraud)]).astype(int)
        idx = self.rng.permutation(n)
        return X[idx], y[idx]

In [ ]:
# Exemplos reais de cada tipo (6 transações de cada)
b = Bank(np.random.default_rng(3))
tipos = {"Cliente normal": b.legit(6, False), "Cliente na Black Friday": b.legit(6, True),
         "Fraude clássica": b.fraud_classic(6), "Fraude nova (teste de cartão)": b.fraud_card_testing(6)}
print(f"{'tipo':32s} {'valor (R$)':>10s} {'madrugada':>10s} {'dist (km)':>10s} {'tx na hora':>11s}")
for nome, X in tipos.items():
    print("-" * 78)
    for r in X[:4]:
        print(f"{nome:32s} {np.exp(r[0]):10.0f} {'sim' if r[1] else 'não':>10s} {np.exp(r[2]):10.1f} {int(r[3]):11d}")

**Em gráfico:** repare que a fraude nova (vermelha) tem valor **menor** que o cliente normal. Um modelo que aprendeu "valor alto = fraude" vai deixá-la passar.

> 🎤 **Fale:** "Cada compra tem quatro características: valor, se foi de madrugada, distância de casa e quantas transações o cartão fez na última hora. Vejam a diferença entre os grupos: na Black Friday os clientes bons ficam parecidos com fraudes do jeito antigo, e a fraude nova tem valor pequeno, o oposto do que o modelo aprendeu."

In [ ]:
b = Bank(np.random.default_rng(0))
grupos = {"Cliente normal": b.legit(4000, False),
          "Cliente na Black Friday": b.legit(4000, True),
          "Fraude clássica": b.fraud_classic(1000),
          "Fraude nova (teste de cartão)": b.fraud_card_testing(1000)}
cores = ["#2ecc71", "#f39c12", "#3498db", "#e74c3c"]

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
for (nome, X), c in zip(grupos.items(), cores):
    axs[0].hist(np.exp(X[:, 0]), bins=np.linspace(0, 1500, 60), alpha=0.55, color=c, label=nome, density=True)
    axs[1].hist(X[:, 3], bins=np.arange(-0.5, 12.5, 1), alpha=0.55, color=c, label=nome, density=True)
axs[0].set_title("Valor da compra (R$)"); axs[0].legend(fontsize=8)
axs[1].set_title("Transações na última hora")
plt.tight_layout(); plt.show()

## 4. Os dois modelos
- **Algoritmo:** regressão logística treinada com SGD (`SGDClassifier`), que permite `partial_fit` (aprender aos poucos).
- **Treino inicial:** só com dados de um **dia normal** (20.000 transações, 6% de fraude).
- **Diferença entre eles:** o **estático** nunca mais é tocado; o **online** chama `partial_fit` a cada lote com rótulos.

> 🎤 **Fale:** "Os dois modelos são idênticos no começo: mesmo algoritmo, mesmo treino, só com dados de um dia normal. A única diferença é que o online continua aprendendo com `partial_fit` e o estático não."

In [ ]:
rng = np.random.default_rng(SEED)
bank = Bank(rng)
tr = [bank.batch(0) for _ in range(80)]
X_tr = np.vstack([t[0] for t in tr]); y_tr = np.concatenate([t[1] for t in tr])
mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-9
scale = lambda X: (X - mu) / sd

def make_model():
    m = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=LR_NORMAL,
                      alpha=1e-4, class_weight={0: 1, 1: 4}, random_state=SEED)
    m.fit(scale(X_tr), y_tr)
    return m

m = make_model()
print(f"Treino com {len(y_tr)} transações normais ({y_tr.mean()*100:.1f}% fraudes)")
print(f"Taxa de bloqueio normal: {m.predict(scale(X_tr)).mean()*100:.1f}%")

## 5. Zoom: o que acontece em **uma** hora
Vamos pegar o modelo **estático** (treinado no dia normal) e ver como ele se sai em **três horas** diferentes. Para cada hora, calculamos:

- **Verdadeiro positivo (VP):** fraude que o modelo bloqueou ✅
- **Falso negativo (FN):** fraude que **passou** ❌ (custa o valor da compra + R$ 120)
- **Falso positivo (FP):** cliente bom **bloqueado** ❌ (custa R$ 30)
- **Recall** = VP / (VP + FN): *de cada 100 fraudes, quantas o modelo pegou*

In [ ]:
estatico = make_model()
bz = Bank(np.random.default_rng(99))

def zoom(step, nome):
    X, y = bz.batch(step); p = estatico.predict(scale(X)); valor = np.exp(X[:, 0])
    vp = int(((y == 1) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum())
    fp = int(((y == 0) & (p == 1)).sum())
    c_fraude = valor[(y == 1) & (p == 0)].sum() + fn * FRAUD_FIXED_COST
    c_fp = fp * FP_COST
    print(f"── Hora {step:3d} · {nome}")
    print(f"   Compras: {len(y)} | fraudes reais: {y.sum()} | bloqueadas pelo modelo: {p.sum()} ({p.mean()*100:.0f}%)")
    print(f"   ✅ Fraudes pegas (VP): {vp:3d}   ❌ Fraudes que passaram (FN): {fn:3d}   ❌ Clientes bons bloqueados (FP): {fp:3d}")
    print(f"   Recall = {vp/max(vp+fn,1)*100:.0f}%")
    print(f"   💰 Prejuízo da hora: fraude que passou R$ {c_fraude:,.0f} + clientes bloqueados R$ {c_fp:,.0f} = R$ {c_fraude+c_fp:,.0f}\n")

zoom(30, "Ato 1 (dia normal)")
zoom(70, "Ato 2 (Black Friday)")
zoom(110, "Ato 3 (fraude nova)")

**Como ler:**
- **Dia normal:** poucas fraudes escapam e poucos clientes são bloqueados → prejuízo baixo.
- **Black Friday:** o modelo bloqueia **muito cliente bom** (FP alto): o prejuízo vem do atrito.
- **Fraude nova:** o recall cai e muitas fraudes **passam** (FN alto): o prejuízo vem do chargeback.

A simulação inteira é a soma disso, hora a hora, para os dois modelos.

> 🎤 **Fale:** "Vamos olhar uma hora de cada ato só com o modelo estático. No dia normal ele vai bem. Na Black Friday ele bloqueia clientes bons: o prejuízo é de atrito. Na fraude nova ele deixa fraudes passarem: o prejuízo é de chargeback. A simulação inteira é a soma disso, hora a hora."

## 6. O detalhe mais realista: o rótulo chega com atraso
Em um banco, a gente só descobre que uma compra era fraude quando o cliente contesta (**chargeback**), horas ou dias depois.
Na simulação, esse atraso é de **4 horas** (`LABEL_DELAY`).

Consequências:
1. O **recall** só pode ser calculado **depois** (por isso é um sinal **lento**).
2. O modelo **online** só pode aprender com uma compra **depois** que o rótulo chega.
3. **PSI** e **taxa de bloqueio** não dependem do rótulo → são sinais **rápidos**.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.8))
for i, t in enumerate([10, 11, 12]):
    y0 = 2 - i
    ax.scatter([t], [y0], s=120, color="#3498db", zorder=3)
    ax.scatter([t + LABEL_DELAY], [y0], s=120, color="#16a085", zorder=3)
    ax.annotate("", xy=(t + LABEL_DELAY - 0.15, y0), xytext=(t + 0.15, y0), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.text(t, y0 + 0.28, f"hora {t}: compras chegam\ne os modelos decidem", ha="center", fontsize=8)
    ax.text(t + LABEL_DELAY, y0 - 0.45, "chargeback chega:\nmede recall; online aprende", ha="center", fontsize=8)
ax.set_xlim(8, 18); ax.set_ylim(-0.9, 3); ax.set_yticks([]); ax.set_xlabel("tempo (horas)"); ax.grid(False)
ax.set_title(f"Atraso do rótulo = {LABEL_DELAY} horas", fontsize=11); plt.show()

> 🎤 **Fale:** "O detalhe mais realista é que o rótulo demora: o banco só descobre que era fraude quando o cliente contesta. Aqui são 4 horas. Por isso o recall é um sinal lento, e por isso o modelo online só aprende com um lote depois que o rótulo chega. Já o PSI e a taxa de bloqueio não precisam do rótulo e aparecem na hora."

## 7. Sinal rápido nº 1: o PSI
Compara a distribuição de cada característica **hoje** (últimas 4 horas) com a do **treino**. **Não precisa de rótulo.**

| PSI | Leitura |
|---|---|
| < 0,1 | Estável |
| 0,1 a 0,2 | Atenção |
| > 0,2 | **Mudou**, dispara alerta |

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(ref, edges)[0] / len(ref)
    q = np.histogram(cur, edges)[0] / len(cur)
    p, q = np.clip(p, 1e-4, None), np.clip(q, 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))

ref = np.random.default_rng(1).normal(0, 1, 5000)
for desc, cur in [("Mesma distribuição", np.random.default_rng(2).normal(0, 1, 1000)),
                  ("Média deslocada um pouco (+0.5)", np.random.default_rng(3).normal(0.5, 1, 1000)),
                  ("Média deslocada muito (+1.5)", np.random.default_rng(4).normal(1.5, 1, 1000))]:
    print(f"{desc:35s} -> PSI = {psi(ref, cur):.2f}")

> 🎤 **Fale:** "O PSI é um número que compara os dados de hoje com os do treino. Zero significa igual; quanto maior, mais diferente. Acima de 0,2 a gente dispara um alerta. A vantagem: não precisa saber se a compra era fraude."

## 8. Os alertas e a ação automática
O sistema vigia o modelo em produção (o **online**) com **3 regras**:

| Regra | Dispara quando | Precisa de rótulo? |
|---|---|---|
| 🟡 **Perfil dos dados mudou** | PSI de alguma característica > 0,20 | Não |
| 🟡 **Taxa de bloqueio anormal** | Bloqueio > 1,8× ou < 1/1,8× do normal (7%) | Não |
| 🔴 **Recall abaixo da meta** | Recall < 70% | **Sim** (atrasado) |

**Ação automática:** quando um alerta novo dispara, o modelo online **aprende mais rápido** (taxa de aprendizado sobe de 0,01 para 0,06) por 10 horas e depois volta ao normal. É a estratégia de *atualização por alerta*.

> 🎤 **Fale:** "O sistema tem três regras de alerta: o perfil dos dados mudou, a taxa de bloqueio ficou estranha, ou o recall caiu abaixo da meta. As duas primeiras não precisam de rótulo; a terceira precisa. Quando um alerta dispara, o modelo online aprende mais rápido por um tempo. Isso é atualização disparada por alerta."

## 9. Rodando a simulação completa
A função `simular()` junta tudo: gera as 150 horas, mede os sinais, dispara os alertas e atualiza o modelo online. **O log abaixo é o diário do sistema.**

In [ ]:
def act_name(step):
    return next(n for s, e, n, _ in ACTS if s <= step < e)

def recall_of(win):
    tp = sum(a for a, _ in win); fn = sum(b for _, b in win)
    return tp / (tp + fn) if (tp + fn) else np.nan

def simular(label_delay=LABEL_DELAY, online_learns=True, verbose=True):
    rng = np.random.default_rng(SEED); bank = Bank(rng)
    for _ in range(80): bank.batch(0)            # mesmo estado do treino
    static, online = make_model(), make_model()
    base_block = float(online.predict(scale(X_tr)).mean())

    pending, win_X, blk_win = deque(), deque(maxlen=WINDOW), deque(maxlen=WINDOW)
    rec_win = {"s": deque(maxlen=WINDOW), "o": deque(maxlen=WINDOW)}
    H = {k: [] for k in ["t", "bs", "bo", "rs", "ro", "cs", "co"]}; H["psi"] = [[] for _ in FEATURES]
    cost = {"s": 0.0, "o": 0.0}; events = []; first = {}
    state = {"psi": False, "rec": False, "blk": False}; boost_left = 0

    def event(step, text):
        events.append((step, text))
        if verbose: print(f"[hora {step:3d}] {text}")

    for step in range(N_STEPS):
        X, y = bank.batch(step); Z = scale(X)
        ps, po = static.predict(Z), online.predict(Z)
        valor = np.exp(X[:, 0])

        # prejuízo (o simulador conhece a verdade; o banco só descobre depois)
        for tag, p in (("s", ps), ("o", po)):
            miss = (y == 1) & (p == 0)
            cost[tag] += valor[miss].sum() + miss.sum() * FRAUD_FIXED_COST + ((y == 0) & (p == 1)).sum() * FP_COST

        # sinais SEM rótulo
        blk_win.append(po.mean()); win_X.append(X)
        Xw = np.vstack(win_X)
        psis = [psi(X_tr[:, j], Xw[:, j]) for j in range(len(FEATURES))]

        # rótulos chegam com atraso -> mede recall e (só o online) aprende
        pending.append((step + label_delay, Z, y, ps, po))
        while pending and pending[0][0] <= step:
            _, Zl, yl, psl, pol = pending.popleft(); pos = yl == 1
            rec_win["s"].append(((psl[pos] == 1).sum(), (psl[pos] == 0).sum()))
            rec_win["o"].append(((pol[pos] == 1).sum(), (pol[pos] == 0).sum()))
            if online_learns: online.partial_fit(Zl, yl)
        rec_s, rec_o = recall_of(rec_win["s"]), recall_of(rec_win["o"])

        # alertas (sobre o modelo em produção = online)
        worst = int(np.argmax(psis))
        psi_bad = max(psis) > PSI_LIMIT and step >= WINDOW
        ratio = float(np.mean(blk_win)) / base_block
        blk_bad = step >= WINDOW and (ratio > BLOCK_RATE_FACTOR or ratio < 1 / BLOCK_RATE_FACTOR)
        rec_bad = (not np.isnan(rec_o)) and rec_o < RECALL_SLO
        new_alert = False
        if psi_bad and not state["psi"]:
            event(step, f"ALERTA: perfil dos dados mudou ({FEATURES[worst]}, PSI={max(psis):.2f})"); first.setdefault("psi", step); new_alert = True
        if not psi_bad and state["psi"]: event(step, "OK: perfil dos dados voltou ao normal")
        if blk_bad and not state["blk"]:
            event(step, f"ALERTA: taxa de bloqueio anormal ({np.mean(blk_win)*100:.0f}% vs {base_block*100:.0f}%)"); first.setdefault("blk", step); new_alert = True
        if not blk_bad and state["blk"]: event(step, "OK: taxa de bloqueio normalizada")
        if rec_bad and not state["rec"]:
            event(step, f"CRÍTICO: recall {rec_o*100:.0f}% abaixo da meta de {RECALL_SLO*100:.0f}%"); first.setdefault("rec", step); new_alert = True
        if (not rec_bad) and state["rec"] and not np.isnan(rec_o) and rec_o > RECALL_SLO + 0.1:
            event(step, f"OK: recall recuperado ({rec_o*100:.0f}%)"); state["rec"] = False
        if rec_bad: state["rec"] = True
        state["psi"], state["blk"] = psi_bad, blk_bad

        if new_alert and boost_left == 0:
            boost_left = BOOST_STEPS; online.set_params(eta0=LR_BOOST)
            event(step, f"AÇÃO: modelo online aprende mais rápido (lr {LR_NORMAL} -> {LR_BOOST})")
        elif boost_left > 0:
            boost_left -= 1
            if boost_left == 0:
                online.set_params(eta0=LR_NORMAL); event(step, "AÇÃO: aprendizado volta ao ritmo normal")

        H["t"].append(step); H["bs"].append(ps.mean() * 100); H["bo"].append(po.mean() * 100)
        H["rs"].append(rec_s); H["ro"].append(rec_o)
        H["cs"].append(cost["s"] / 1000); H["co"].append(cost["o"] / 1000)
        for j in range(len(FEATURES)): H["psi"][j].append(psis[j])
    return dict(H=H, cost=cost, events=events, first=first)

R = simular()

**Leia o log:**
- **Hora 51:** o PSI e a taxa de bloqueio acusam a Black Friday (sem usar rótulo) → o online aprende mais rápido.
- **Hora 92–94:** a fraude nova derruba o bloqueio e depois o recall abaixo de 70% (alerta **crítico**, já com rótulos).
- **Hora 102:** o recall se recupera (97%) → o sistema se normaliza.

> 🎤 **Fale:** "Esse é o diário do sistema. Na hora 51 o PSI avisou da Black Friday, sem rótulo. Na hora 94, o recall confirmou que a fraude nova estava passando, e o alerta crítico disparou. Na hora 102 o modelo online já tinha se recuperado."

## 10. Os resultados em gráficos

In [ ]:
def fundo(ax):
    for s, e, _, col in ACTS: ax.axvspan(s, e, color=col, alpha=0.10, lw=0)
    for s, e, nome, _ in ACTS: ax.text((s + e) / 2, 1.01, nome, transform=ax.get_xaxis_transform(), ha="center", fontsize=8)
    ax.set_xlim(0, N_STEPS); ax.set_xlabel("tempo (horas)")
H = R["H"]

### 10.1 Quanto estamos bloqueando? *(sem rótulo)*
De cada 100 compras, quantas foram bloqueadas. Normal ≈ 7%.

> 🎤 **Fale:** "Na Black Friday, o vermelho sobe para 20%: o estático bloqueia clientes bons. Na fraude nova, ele cai para perto de zero: deixou de bloquear."

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
ax.plot(H["t"], H["bs"], color=C_S, lw=2, label="Estático"); ax.plot(H["t"], H["bo"], color=C_O, lw=2, label="Online")
ax.axhline(7, color="k", ls=":", lw=1); ax.text(1, 8.5, "normal ≈ 7%", fontsize=8)
ax.set_ylabel("% bloqueadas"); ax.legend(loc="upper left"); plt.show()

### 10.2 Fraudes capturadas, o recall *(com rótulo, atrasado)*
De cada 100 fraudes, quantas o modelo pegou. A linha amarela é a meta de 70%.

> 🎤 **Fale:** "Aqui está o problema real: com a fraude nova, os dois despencam. O estático nunca se recupera, o online volta em poucas horas."

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
ax.plot(H["t"], H["rs"], color=C_S, lw=2, label="Estático"); ax.plot(H["t"], H["ro"], color=C_O, lw=2, label="Online")
ax.axhline(RECALL_SLO, color="#d4a017", ls="--"); ax.text(1, RECALL_SLO + 0.02, "meta mínima (SLO) 70%", fontsize=8, color="#d4a017")
ax.set_ylabel("recall"); ax.set_ylim(0, 1.05); ax.legend(loc="lower left"); plt.show()

### 10.3 PSI *(sem rótulo)*
> 🎤 **Fale:** "O PSI explode na Black Friday, mas quase não reage à fraude nova, porque ela é só 8% das compras. Por isso o PSI sozinho não basta."

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
for j, f in enumerate(FEATURES): ax.plot(H["t"], H["psi"][j], lw=1.8, label=f)
ax.axhline(PSI_LIMIT, color="#d4a017", ls="--"); ax.set_ylabel("PSI"); ax.legend(loc="upper left", ncol=2, fontsize=8); plt.show()

### 10.4 Prejuízo acumulado *(a métrica de negócio)*
> 🎤 **Fale:** "E aqui a diferença aparece em dinheiro: R$ 219 mil contra R$ 94 mil."

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
ax.plot(H["t"], H["cs"], color=C_S, lw=2.4, label="Estático"); ax.plot(H["t"], H["co"], color=C_O, lw=2.4, label="Online")
ax.set_ylabel("R$ mil"); ax.legend(loc="upper left")
ax.text(0.98, 0.05, f"Estático: R$ {R['cost']['s']/1000:,.0f} mil\nOnline: R$ {R['cost']['o']/1000:,.0f} mil",
        transform=ax.transAxes, ha="right", fontweight="bold"); plt.show()

### 10.5 Tudo junto: a linha do tempo dos alertas
Uma figura-resumo marcando **quando cada sinal avisou**. Note a diferença de velocidade entre sinal *sem rótulo* e *com rótulo*.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4)); fundo(ax)
ax.plot(H["t"], H["rs"], color=C_S, lw=2, label="Recall estático"); ax.plot(H["t"], H["ro"], color=C_O, lw=2, label="Recall online")
ax.axhline(RECALL_SLO, color="#d4a017", ls="--", lw=1)
marcas = {"ALERTA: perfil": ("PSI avisa\n(sem rótulo)", "#f39c12", 0.30),
          "CRÍTICO": ("recall confirma\n(com rótulo)", "#c0392b", 0.15),
          "recall recuperado": ("online\nrecuperado", "#16a085", 0.45)}
for step, texto in R["events"]:
    for chave, (rotulo, cor, y) in marcas.items():
        if chave in texto:
            ax.axvline(step, color=cor, ls=":", lw=1.6)
            ax.text(step + 1, y, f"{rotulo}\nhora {step}", fontsize=8, color=cor, fontweight="bold")
ax.set_ylim(0, 1.05); ax.set_ylabel("recall"); ax.legend(loc="lower left"); plt.show()

> 🎤 **Fale:** "Resumindo no tempo: o PSI avisou na hora 51, sem esperar rótulo. O recall só confirmou o problema da fraude nova na hora 94, porque dependia do chargeback. E na hora 102 o modelo online já estava recuperado."

## 11. Resumo numérico

In [ ]:
c = R["cost"]; eco = c["s"] - c["o"]
print(f"Prejuízo acumulado | estático: R$ {c['s']:,.0f}  |  online: R$ {c['o']:,.0f}")
print(f"Economia por atualizar online: R$ {eco:,.0f} ({eco / c['s'] * 100:.0f}%)")
for k, nome in (("psi", "mudança de perfil (PSI)"), ("blk", "taxa de bloqueio"), ("rec", "queda do recall")):
    if k in R["first"]: print(f"Primeiro alerta de {nome}: hora {R['first'][k]}")

## 12. Prova de que a atualização é o que faz diferença
Se o sistema só **observasse** (alertas) mas o modelo **não aprendesse**, o prejuízo seria igual ao do estático.

In [ ]:
# E se o modelo online NÃO aprendesse (só observabilidade, sem atualização)?
R0 = simular(online_learns=False, verbose=False)
print(f"Só observabilidade, sem atualizar: R$ {R0['cost']['o']:,.0f}   (alertas disparam, mas ninguém corrige)")

# E se os rótulos demorassem muito mais (chargeback em 12 horas)?
R12 = simular(label_delay=12, verbose=False)
print(f"Rótulos com 12h de atraso:        R$ {R12['cost']['o']:,.0f}   (corrige, mas mais devagar)")
print(f"Referência (atraso 4h):           R$ {R['cost']['o']:,.0f}")

> 🎤 **Fale:** "Se o sistema só observar e não atualizar, o prejuízo continua R$ 219 mil: os alertas disparam, mas ninguém corrige. E se os rótulos demorarem mais, o modelo corrige, só que mais devagar."

## 13. O que é simplificado nesta simulação (seja honesto no vídeo)
| Na simulação | Na vida real |
|---|---|
| Dados **gerados por regras** simples | Dados reais, bagunçados, com muito mais variáveis |
| Drift **abrupto** e com hora marcada | Drift costuma ser gradual e difícil de datar |
| Rótulos **sempre chegam**, com atraso fixo de 4 h | Atraso variável; alguns rótulos nunca chegam |
| Modelo **linear simples** | Modelos mais complexos (árvores, redes) |
| Valores em R$ **inventados** | Custos reais da operação |
| Sem validação antes de atualizar | Em produção, valida-se (shadow/canary) e há *rollback* |

> **O que vale é o conceito:** sinais sem rótulo avisam cedo, sinais com rótulo confirmam, e atualizar o modelo corrige.

> 🎤 **Fale:** "Para ser honesto: é uma simulação simplificada, com dados gerados por regras e um drift abrupto. Na vida real é mais bagunçado. O que a simulação mostra é o conceito: observabilidade avisa e atualização corrige."

## 14. (Opcional) Reprodução "ao vivo" do dashboard
Para fechar o vídeo mostrando tudo acontecendo hora a hora:

In [ ]:
from IPython.display import clear_output
import time

def ao_vivo(passo=3, pausa=0.03):
    for n in list(range(passo, N_STEPS, passo)) + [N_STEPS]:
        fig, axs = plt.subplots(2, 2, figsize=(12, 6.2))
        t = H["t"][:n]
        for ax in axs.flat: fundo(ax)
        axs[0, 0].plot(t, H["bs"][:n], color=C_S); axs[0, 0].plot(t, H["bo"][:n], color=C_O); axs[0, 0].set_title("Taxa de bloqueio (%)", fontsize=10); axs[0, 0].set_ylim(0, 60)
        axs[0, 1].plot(t, H["rs"][:n], color=C_S); axs[0, 1].plot(t, H["ro"][:n], color=C_O); axs[0, 1].axhline(RECALL_SLO, color="#d4a017", ls="--"); axs[0, 1].set_title("Recall (fraudes capturadas)", fontsize=10); axs[0, 1].set_ylim(0, 1.05)
        for j in range(len(FEATURES)): axs[1, 0].plot(t, H["psi"][j][:n], lw=1.5)
        axs[1, 0].axhline(PSI_LIMIT, color="#d4a017", ls="--"); axs[1, 0].set_title("PSI (mudança nos dados)", fontsize=10); axs[1, 0].set_ylim(0, 2.5)
        axs[1, 1].plot(t, H["cs"][:n], color=C_S, label="Estático"); axs[1, 1].plot(t, H["co"][:n], color=C_O, label="Online"); axs[1, 1].set_title("Prejuízo acumulado (R$ mil)", fontsize=10); axs[1, 1].legend(loc="upper left")
        fig.suptitle(act_name(n - 1), fontweight="bold"); fig.tight_layout()
        clear_output(wait=True); plt.show(); time.sleep(pausa)

ao_vivo()